In [1]:
import mbuild as mb
from mbuild import Compound

# Ignore these imports, this is just to prevent warning messages for the purposes of the workshop
from mbuild import mBuildLogger
import logging
mbuild_logger = mBuildLogger()
mbuild_logger.library_logger.setLevel(logging.ERROR)

In [2]:
# Create molecules

In [3]:
benzene_smiles = mb.load("c1ccccc1", smiles=True)
benzene_file = mb.load("../files/benzene.mol2")

In [4]:
print(type(benzene_smiles))
print(type(benzene_file))

<class 'mbuild.compound.Compound'>
<class 'mbuild.compound.Compound'>


In [5]:
benzene_smiles.visualize().show()
benzene_file.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
class DumbBell(Compound):
    def __init__(self, bond_length, A_name, B_name):
        super(DumbBell, self).__init__()
        bead_A = Compound(name=A_name, pos=(0,0,0))
        bead_B = Compound(name=B_name, pos=(bond_length, 0, 0))
        self.add(bead_A)
        self.add(bead_B)
        self.add_bond([bead_A, bead_B])

In [7]:
dumb_bell = DumbBell(bond_length=0.15, A_name="_A", B_name="_B")
dumb_bell.visualize(color_scheme={"_A": "red", "_B": "blue"})

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
class FattyAcid(Compound):
    SMILES = {
        "cis":   "CCCCCCCC/C=C\\CCCCCCCC(=O)O",   # oleic acid
        "trans": "CCCCCCCC/C=C/CCCCCCCC(=O)O",    # elaidic acid
    }
    def __init__(self, isomer="cis"):
        super(FattyAcid, self).__init__()
        mb.load(self.SMILES[isomer], smiles=True, compound=self)

In [9]:
fa_cis = FattyAcid("cis")
fa_trans = FattyAcid("trans")

fa_cis.visualize().show()
fa_trans.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
oleic = FattyAcid(isomer="cis")
oleic.name = "FA"
elaidic = FattyAcid(isomer="trans")
elaidic.name = "FA"
water = mb.load("O", smiles=True)
water.name = "Water"

mixture = mb.packing.fill_box(
    compound=[
        water,
        oleic,
        elaidic
    ],
    n_compounds=[
        200,
        20,
        20
    ],
    box=[4, 4, 4],
)
mixture.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## General Molecular Simulation Object (GMSO)

- Graph-based atom typing: SMARTS string `[C;X3]([O;X1])(C)(C)(H)` → Molecular Graph (networkX)
- Uses and XML scheme to define:
    - Analytical forms of forces and parameters: $E(r) = 4\epsilon\left[\left(\dfrac{\sigma}{r}\right)^{12} - \left(\dfrac{\sigma}{r}\right)^6\right]$
    - Parameter values and units: $\sigma=0.35; \epsilon=0.875 \frac{kJ}{mol}$
    - Atom type definitions: `[C;X3]([O;X1])([O;X2])(C)`
    - Atom types and Atom classes: `ca` 

In [29]:
import gmso
from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

In [30]:
gaff = ForceField("../files/gaff.xml")
spce = ForceField("../files/spcfw.xml")

In [13]:
mixture_topology = mixture.to_gmso()

In [14]:
apply(
    top=mixture_topology,
    identify_connections=True,
    forcefields={"FA": gaff, "Water": spce},
    speedup_by_moltag=True
)

<Topology Topology, 2760 sites,
 12400 connections,
 15160 potentials,
 id: 6435597552>

In [26]:
for particle in [p for p in mixture.particles()][:10]:
    print(particle.name)

O
H
H
O
H
H
O
H
H
O


In [28]:
for site in mixture_topology.sites[:10]:
    print(site.atom_type.name, site.atom_type.parameters, site.atom_type.definition)

print()

for site in mixture_topology.sites[600:610]:
    print(site.atom_type.name, site.atom_type.parameters, site.atom_type.definition)
    

OW {'epsilon': unyt_quantity(0.650194, 'kJ/mol'), 'sigma': unyt_quantity(0.316557, 'nm')} O
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0.1, 'nm')} H
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0.1, 'nm')} H
OW {'epsilon': unyt_quantity(0.650194, 'kJ/mol'), 'sigma': unyt_quantity(0.316557, 'nm')} O
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0.1, 'nm')} H
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0.1, 'nm')} H
OW {'epsilon': unyt_quantity(0.650194, 'kJ/mol'), 'sigma': unyt_quantity(0.316557, 'nm')} O
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0.1, 'nm')} H
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0.1, 'nm')} H
OW {'epsilon': unyt_quantity(0.650194, 'kJ/mol'), 'sigma': unyt_quantity(0.316557, 'nm')} O

c3 {'epsilon': unyt_quantity(0.4577296, 'kJ/mol'), 'sigma': unyt_quantity(0.33996695, 'nm')} [C;X4]
c3 {'epsilon': unyt_quantity(0.4577296, 'kJ/m